In [1]:
import os

# TO DO: manually edit file name, file type, worksheet name, and columns you'd use for geocoding a location.

FILE_NAME = "Landfills.csv"
MIME_TYPE = "text/csv"
WORKSHEET_NAME = "Landfills"
address_columns_sans_latlong = ["Landfill Name", "State", "Physical Address", "City", "County", "Zip Code"]
# Please do lat first, then long
latlong_columns_names = ["Latitude", "Longitude"]
# if all points are in CA
state_is_ca = False



# DO NOT CHANGE THESE.
# -------------------------------------------------------------------------------------------------------------------------------------------------------
ADDRESS_COLUMNS = address_columns_sans_latlong + latlong_columns_names
GSHEET_NAME = "address-to-geocoded" # look for this spreadsheet in Google Drive
geocoded_columns = ["status", "closest_address_line_1", "closest_address_line_2", "closest_city", "closest_county", "closest_state", "closest_postal_code", "closest_latitude", "closest_longitude", "closest_geoid", "closest_state_name", "closest_state_fips", "closest_county_name", "closest_county_fips","is_na","address_id", "merged_address"]
CREDENTIALS_PATH = "../../../../../../resources/prefect/credentials.json"
DATASET_FOLDER = "../temp_external_datasets/"

ADDRESS_COLUMNS = [col.lower().replace(" ", "_") for col in ADDRESS_COLUMNS]
latlong_columns_names = [col.lower().replace(" ", "_") for col in latlong_columns_names]
address_columns_sans_latlong = [col.lower().replace(" ", "_") for col in address_columns_sans_latlong]

# Determine project root for robust path discovery
PROJECT_ROOT = os.environ.get("CA_BIOSITING_PROJECT_ROOT", "../../../../../../")
CREDENTIALS_PATHS = [
    os.path.join(PROJECT_ROOT, "credentials.json"),
    os.path.join(PROJECT_ROOT, "resources/prefect/credentials.json"),
    "credentials.json"
]
CREDENTIALS_PATH = next((p for p in CREDENTIALS_PATHS if os.path.exists(p)), CREDENTIALS_PATHS[0])
DATASET_FOLDER = os.path.join(PROJECT_ROOT, "src/ca_biositing/pipeline/ca_biositing/pipeline/temp_external_datasets/")

In [2]:
# TO DO: just run this code snippet. Extracts raw csv/zip/geojson file

import os
import pandas as pd
import gspread
from gspread.exceptions import WorksheetNotFound
from ca_biositing.pipeline.utils.gdrive_to_pandas import gdrive_to_df

print(f"Extracting raw data from '{FILE_NAME}'...")

credentials_path = CREDENTIALS_PATH
dataset_folder = DATASET_FOLDER

if os.path.exists(FILE_NAME):
    print(f"Found local file at '{FILE_NAME}'. Loading locally for testing...")
    new_data_df = pd.read_csv(FILE_NAME)
else:
    print(f"Local file not found. Attempting to pull '{FILE_NAME}' from Google Drive using credentials at {CREDENTIALS_PATH}...")
    new_data_df = gdrive_to_df(FILE_NAME, MIME_TYPE, credentials_path, dataset_folder)

if new_data_df is None:
    print("Failed to extract data. Aborting.")

print("Successfully extracted raw data.")

Extracting raw data from 'Landfills.csv'...
Local file not found. Attempting to pull 'Landfills.csv' from Google Drive using credentials at ../../../../../../credentials.json...
Successfully extracted raw data.


In [3]:
# TO DO: just run this code snippet.
# tests whether the worksheet exists; if not, it creates a new one
gc = gspread.service_account(filename=CREDENTIALS_PATH)
sh = gc.open(GSHEET_NAME)
try:
    worksheet = sh.worksheet(WORKSHEET_NAME)
except WorksheetNotFound:
    # Create new worksheet if it doesn't exist yet
    worksheet = sh.add_worksheet(title=WORKSHEET_NAME, rows=len(new_data_df)+1, cols=len(geocoded_columns)+len(ADDRESS_COLUMNS))
    columns = [ADDRESS_COLUMNS+geocoded_columns]
    worksheet.update(columns)


# Geocoder Module

In [4]:
# TO DO: just run this code snippet once -- it creates an extractor for the gsheet
from ca_biositing.pipeline.etl.extract.factory import create_extractor

extract = create_extractor(GSHEET_NAME, WORKSHEET_NAME)


In [5]:
# TO DO: just run this code snippet. extracts the gsheet
gsheet_df = extract(os.path.join(PROJECT_ROOT, "resources/prefect/"))

07:12:10.800 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8012
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

07:12:15.356 | INFO    | Task run 'extract_landfills' - Extracting raw data from 'Landfills' in 'address-to-geocoded'...

DEBUG: gsheet_to_df called for address-to-geocoded / Landfills
DEBUG: Authenticating with ../../../../../../resources/prefect/credentials.json
DEBUG: Opening spreadsheet address-to-geocoded
DEBUG: Opening worksheet by name: Landfills
DEBUG: Fetching all values from worksheet
DEBUG: Successfully fetched 1 rows


07:12:16.670 | INFO    | Task run 'extract_landfills' - Successfully extracted raw data from Landfills.

/workspaces/ca-biositing/.pixi/envs/default/lib/python3.13/logging/__init__.py:1946: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


07:12:16.674 | INFO    | Task run 'extract_landfills' - Finished in state Completed()

In [6]:
# make gsheet lowercase
from ca_biositing.pipeline.utils.cleaning_functions import cleaning as cleaning_mod, clean_names_df
gsheet_df = clean_names_df(gsheet_df)
gsheet_df

,landfill_name,state,physical_address,city,county,zip_code,latitude,longitude,status,closest_address_line_1,...,closest_longitude,closest_geoid,closest_state_name,closest_state_fips,closest_county_name,closest_county_fips,is_na,address_id,merged_address,address


In [7]:
# make incoming data lowercase to standardize capitalization
new_data_df = cleaning_mod.standard_clean(new_data_df)
new_data_df

/workspaces/ca-biositing/src/ca_biositing/pipeline/ca_biositing/pipeline/utils/cleaning_functions/cleaning.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.astype("object").replace(regex, np.nan, regex=True)


,ghgrp_id,landfill_id,landfill_name,state,physical_address,city,county,zip_code,latitude,longitude,...,project_start_date,project_shutdown_date,project_type_category,lfg_energy_project_type,rng_delivery_method,actual_mw_generation,rated_mw_capacity,lfg_flow_to_project_mmscfd_,current_year_emission_reductions_mmtco2e_yr_direct,current_year_emission_reductions_mmtco2e_yr_avoided
0,1007192,645,allen county lf,ks,1170 2800th street,la harpe,allen,66751,37.90316,-95.2808,...,1/1/07,<NA>,direct,direct thermal,<NA>,<NA>,<NA>,0.0504,0.0049,0.0004
1,1003826,1595,atlantic waste disposal inc.,va,3474 atlantic lane,waverly,sussex,23890,37.06925,-77.18108,...,12/31/21,<NA>,direct,leachate evaporation,<NA>,<NA>,<NA>,2.88,0.2824,0.0249
2,1006132,2068,autumn hills recycling & disposal facility,mi,700 56th avenue,zeeland,ottawa,49464,42.781,-85.9207,...,1/1/05,<NA>,direct,boiler,<NA>,<NA>,<NA>,0.5,0.049,0.0043
3,1006969,940,burlington county slf,nj,22000 burlington-columbus road,florence,burlington,8022,40.076,-74.76,...,1/1/96,<NA>,direct,greenhouse,<NA>,<NA>,<NA>,0.216,0.0212,0.0019
4,1004341,2062,cerro colorado lf,nm,18000 cerro colorado sw,albuquerque,bernalillo,87121,35.0105,-106.877,...,6/13/13,<NA>,direct,boiler,<NA>,<NA>,<NA>,0.25,0.0245,0.0022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536,1007984,2199,upper piedmont regional landfill,nc,9650 oxford road,rougemont,person,27572,36.324,-78.832,...,9/12/23,<NA>,renewable natural gas,vehicle fuel,pipeline injection,<NA>,<NA>,<NA>,0.0,0.0
537,1007692,1285,valley lf,pa,6015 pleasant valley road,irwin,westmoreland,15642,40.3881,-79.6742,...,2/27/04,<NA>,renewable natural gas,vehicle fuel,pipeline injection,<NA>,<NA>,1.87,0.1834,0.0148
538,1000291,11721,westmoreland waste slf llc,pa,900 tyrol boulevard\n111 conner lane,belle vernon,westmoreland,15012,40.1443,-79.8593,...,5/11/20,<NA>,renewable natural gas,vehicle fuel,local use,<NA>,<NA>,1.553,0.1523,0.0123
539,1003582,1912,woodland meadows landfill,mi,5900 hannan road\n39900 van born road (canton),wayne,wayne,48184,42.2652,-83.4315,...,4/23/18,<NA>,renewable natural gas,vehicle fuel,pipeline injection,<NA>,<NA>,7.76,0.7608,0.0612


In [8]:
# TO DO: just run this code snippet.

# get unique rows from incoming dataset that aren't in google sheet
gsheet_df_address = gsheet_df[ADDRESS_COLUMNS]
new_data_df_address = new_data_df[ADDRESS_COLUMNS]

# This compares the incoming data to the data already in the Google Sheet
# And returns new rows
comparison_df = new_data_df_address.merge(gsheet_df_address.drop_duplicates(), on=ADDRESS_COLUMNS, 
                   how='left', indicator=True)
new_rows = comparison_df[comparison_df['_merge'] == 'left_only'][ADDRESS_COLUMNS]
new_rows["status"] = "pending"
if state_is_ca:
    new_rows["state"] = "CA"
new_rows = cleaning_mod.standard_clean(new_rows)
new_rows
# add rows to google sheet df and list them as pending

/workspaces/ca-biositing/src/ca_biositing/pipeline/ca_biositing/pipeline/utils/cleaning_functions/cleaning.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.astype("object").replace(regex, np.nan, regex=True)


,landfill_name,state,physical_address,city,county,zip_code,latitude,longitude,status
0,allen county lf,ks,1170 2800th street,la harpe,allen,66751,37.90316,-95.2808,pending
1,atlantic waste disposal inc.,va,3474 atlantic lane,waverly,sussex,23890,37.06925,-77.18108,pending
2,autumn hills recycling & disposal facility,mi,700 56th avenue,zeeland,ottawa,49464,42.781,-85.9207,pending
3,burlington county slf,nj,22000 burlington-columbus road,florence,burlington,8022,40.076,-74.76,pending
4,cerro colorado lf,nm,18000 cerro colorado sw,albuquerque,bernalillo,87121,35.0105,-106.877,pending
...,...,...,...,...,...,...,...,...,...
536,upper piedmont regional landfill,nc,9650 oxford road,rougemont,person,27572,36.324,-78.832,pending
537,valley lf,pa,6015 pleasant valley road,irwin,westmoreland,15642,40.3881,-79.6742,pending
538,westmoreland waste slf llc,pa,900 tyrol boulevard\n111 conner lane,belle vernon,westmoreland,15012,40.1443,-79.8593,pending
539,woodland meadows landfill,mi,5900 hannan road\n39900 van born road (canton),wayne,wayne,48184,42.2652,-83.4315,pending


In [9]:
# TO DO: just run this code snippet. Concats new rows to gsheet_df
gsheet_df = pd.concat([gsheet_df, new_rows])

/tmp/ipykernel_72032/4000224940.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gsheet_df = pd.concat([gsheet_df, new_rows])


In [10]:
gsheet_df

,landfill_name,state,physical_address,city,county,zip_code,latitude,longitude,status,closest_address_line_1,...,closest_longitude,closest_geoid,closest_state_name,closest_state_fips,closest_county_name,closest_county_fips,is_na,address_id,merged_address,address
0,allen county lf,ks,1170 2800th street,la harpe,allen,66751,37.90316,-95.2808,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,atlantic waste disposal inc.,va,3474 atlantic lane,waverly,sussex,23890,37.06925,-77.18108,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,autumn hills recycling & disposal facility,mi,700 56th avenue,zeeland,ottawa,49464,42.781,-85.9207,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,burlington county slf,nj,22000 burlington-columbus road,florence,burlington,8022,40.076,-74.76,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cerro colorado lf,nm,18000 cerro colorado sw,albuquerque,bernalillo,87121,35.0105,-106.877,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536,upper piedmont regional landfill,nc,9650 oxford road,rougemont,person,27572,36.324,-78.832,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
537,valley lf,pa,6015 pleasant valley road,irwin,westmoreland,15642,40.3881,-79.6742,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
538,westmoreland waste slf llc,pa,900 tyrol boulevard\n111 conner lane,belle vernon,westmoreland,15012,40.1443,-79.8593,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
539,woodland meadows landfill,mi,5900 hannan road\n39900 van born road (canton),wayne,wayne,48184,42.2652,-83.4315,pending,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# TO DO: just run this code snippet. geocodes addresses
from ca_biositing.pipeline.utils.geo_utils import parse_addresses
from dotenv import load_dotenv
import pandas as pd

load_dotenv(os.path.join(PROJECT_ROOT, "resources/docker/.env"))

address_df, geoid_df = parse_addresses(
        gsheet_df,
        merge_columns=address_columns_sans_latlong,
        lat=latlong_columns_names[0],
        long=latlong_columns_names[1],
    )

added_address_df = pd.concat([address_df, geoid_df], axis=1)
added_address_df

07:13:11.307 | WARNING | ca_biositing.pipeline.utils.geo_utils - Some addresses were unparsable: 
61      spsa regional lf, va, 1 bob foeller drive, suffolk, suffolk city, 23434
86      arbor hills landfill, mi, 10690 w. six mile road, northville, washtenaw, 48168
87      arbor hills landfill, mi, 10690 w. six mile road, northville, washtenaw, 48168
89      arecibo landfill, pr, carr. pr-682 interior sector garrochales  bo. factor ii, arecibo, arecibo, 612
100     bethel lf, va, 100 north park lane, hampton, hampton city, 23666
328     pennsauken slf, nj, 9600 river road, pennsauken, camden, 8110
348     quarantine road slf, md, 6100 quarantine road, curtis bay, baltimore city, 21226
404     toa baja lf, pr, road 865, bo. candelaria ward
nw of sr #866 & de diego hwy # pr 22, toa baja, toa baja, 961
423     virginia beach landfill no. 2, va, 1989 jake sears road, virginia beach, virginia beach city, 23464
441     arbor hills landfill, mi, 10690 w. six mile road, northville, washtenaw, 48168
524     spsa regional lf, va, 1 bob foeller drive, suffolk, suffolk city, 23434

,status,closest_address_line_1,closest_address_line_2,closest_city,closest_county,closest_state,closest_postal_code,closest_latitude,closest_longitude,closest_geoid,closest_state_name,closest_state_fips,closest_county_name,closest_county_fips
0,true,1170 2800th Street,None,La Harpe,Allen County,KS,66751,37.902764,-95.281695,20001,KS,20,Allen County,001
1,true,3474 Atlantic Lane,None,Waverly,Sussex County,VA,23890,37.068105,-77.178875,51183,VA,51,Sussex County,183
2,true,700 56th Avenue,None,Zeeland,Ottawa County,MI,49464,42.781086,-85.918404,26139,MI,26,Ottawa County,139
3,true,22000 Burlington Columbus Road,None,Florence,Burlington County,NJ,08518,40.076129,-74.768444,34005,NJ,34,Burlington County,005
4,true,,None,Albuquerque,Bernalillo County,NM,None,35.084386,-106.650422,35001,NM,35,Bernalillo County,001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536,true,9650 Oxford Road,None,Rougemont,Person County,NC,27572,36.329235,-78.833407,37145,NC,37,Person County,145
537,true,6015 Pleasant Valley Road,None,Irwin,Westmoreland County,PA,15642,40.388741,-79.671260,42129,PA,42,Westmoreland County,129
538,true,111 Conner Lane,None,Belle Vernon,Westmoreland County,PA,15012-4569,40.145775,-79.857063,42129,PA,42,Westmoreland County,129
539,true,5900 Hannan Road,None,Wayne,Wayne County,MI,48184,42.264880,-83.433672,26163,MI,26,Wayne County,163


In [12]:
# TO DO: just run this code snippet. adds geocoded data to gsheet_df
replace_columns = added_address_df.columns
gsheet_df.loc[gsheet_df["status"] == "pending", replace_columns] = added_address_df
gsheet_df['closest_county_name'] = gsheet_df['closest_county_name'].str.replace(" County", "")
gsheet_df['closest_county'] = gsheet_df['closest_county'].str.replace(" County", "")

In [13]:
# TO DO: just run this code snippet. finds addresses marked "true" to load into LocationAddress database
found_addresses = gsheet_df[gsheet_df["status"] == "true"]
found_addresses

,landfill_name,state,physical_address,city,county,zip_code,latitude,longitude,status,closest_address_line_1,...,closest_longitude,closest_geoid,closest_state_name,closest_state_fips,closest_county_name,closest_county_fips,is_na,address_id,merged_address,address
0,allen county lf,ks,1170 2800th street,la harpe,allen,66751,37.90316,-95.2808,true,1170 2800th Street,...,-95.281695,20001,KS,20,Allen,001,False,NaN,NaN,"allen county lf, ks, 1170 2800th street, la ha..."
1,atlantic waste disposal inc.,va,3474 atlantic lane,waverly,sussex,23890,37.06925,-77.18108,true,3474 Atlantic Lane,...,-77.178875,51183,VA,51,Sussex,183,False,NaN,NaN,"atlantic waste disposal inc., va, 3474 atlanti..."
2,autumn hills recycling & disposal facility,mi,700 56th avenue,zeeland,ottawa,49464,42.781,-85.9207,true,700 56th Avenue,...,-85.918404,26139,MI,26,Ottawa,139,False,NaN,NaN,"autumn hills recycling & disposal facility, mi..."
3,burlington county slf,nj,22000 burlington-columbus road,florence,burlington,8022,40.076,-74.76,true,22000 Burlington Columbus Road,...,-74.768444,34005,NJ,34,Burlington,005,False,NaN,NaN,"burlington county slf, nj, 22000 burlington-co..."
4,cerro colorado lf,nm,18000 cerro colorado sw,albuquerque,bernalillo,87121,35.0105,-106.877,true,,...,-106.650422,35001,NM,35,Bernalillo,001,False,NaN,NaN,"cerro colorado lf, nm, 18000 cerro colorado sw..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536,upper piedmont regional landfill,nc,9650 oxford road,rougemont,person,27572,36.324,-78.832,true,9650 Oxford Road,...,-78.833407,37145,NC,37,Person,145,False,NaN,NaN,"upper piedmont regional landfill, nc, 9650 oxf..."
537,valley lf,pa,6015 pleasant valley road,irwin,westmoreland,15642,40.3881,-79.6742,true,6015 Pleasant Valley Road,...,-79.67126,42129,PA,42,Westmoreland,129,False,NaN,NaN,"valley lf, pa, 6015 pleasant valley road, irwi..."
538,westmoreland waste slf llc,pa,900 tyrol boulevard\n111 conner lane,belle vernon,westmoreland,15012,40.1443,-79.8593,true,111 Conner Lane,...,-79.857063,42129,PA,42,Westmoreland,129,False,NaN,NaN,"westmoreland waste slf llc, pa, 900 tyrol boul..."
539,woodland meadows landfill,mi,5900 hannan road\n39900 van born road (canton),wayne,wayne,48184,42.2652,-83.4315,true,5900 Hannan Road,...,-83.433672,26163,MI,26,Wayne,163,False,NaN,NaN,"woodland meadows landfill, mi, 5900 hannan roa..."


In [14]:
# TO DO: just run this code snippet. Loads addresses and FIPS codes to databases

print("Bridging County (Place) to LocationAddress...")
from sqlmodel import Session, select, create_engine
from ca_biositing.pipeline.utils.engine import _get_database_url, get_engine
from ca_biositing.datamodels.models import LocationAddress, Place
from ca_biositing.datamodels.config import settings

db_url = _get_database_url()
db_url = db_url.replace("@db:5432", f"@localhost:{settings.POSTGRES_PORT}")
db_url = db_url.replace("@db:", f"@localhost:{settings.POSTGRES_PORT}")
engine = create_engine(
        db_url,
        pool_size=5,
        max_overflow=0,
        pool_pre_ping=True,
        connect_args={"connect_timeout": 10}
)

with Session(engine) as session:
    place_to_address_map = {}

    for index, row in found_addresses.iterrows():
        geoid = row.get("closest_geoid")
        if geoid != 00000 and geoid is not None:
            stmt1 = select(Place).where(Place.geoid == geoid)
            place = session.exec(stmt1).first()

            stmt2 = select(LocationAddress).where(
                LocationAddress.geography_id == geoid
            )
            address = session.exec(stmt2).first()

            if not place:
                place = Place(
                    geoid=geoid,
                    state_name=row.get("closest_state_name"),
                    state_fips=row.get("closest_state_fips"),
                    county_name=row.get("closest_county_name"),
                    county_fips=row.get("closest_county_fips"),
                )
                session.add(place)
                session.flush()

            if not address:
                address = LocationAddress(
                    geography_id=geoid,
                    address_line1=row.get("closest_address_line_1"),
                    address_line2=row.get("closest_address_line_2"),
                    city=row.get("closest_city"),
                    zip=row.get("closest_postal_code"),
                    lat=row.get("closest_latitude"),
                    lon=row.get("closest_longitude"),
                    is_anonymous=False,
                )
                session.add(address)
                session.flush()

            place_to_address_map[geoid] = address.id

    session.commit()
    found_addresses["address_id"] = found_addresses["closest_geoid"].map(
        place_to_address_map
    )
    print(
        f"Mapped {len(place_to_address_map)} counties to LocationAddresses"
    )

Bridging County (Place) to LocationAddress...
Mapped 328 counties to LocationAddresses


In [15]:
# TO DO: just run this code snippet. Sets address_ids to ids assigned in database
gsheet_df['address_id'] = 0
gsheet_df.loc[gsheet_df["status"] == "true", "address_id"] = found_addresses["address_id"]
gsheet_df = gsheet_df.drop_duplicates(keep="first")
gsheet_df['merged_address'] = gsheet_df['merged_address'].str.lower()

/tmp/ipykernel_72032/860121839.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gsheet_df['merged_address'] = gsheet_df['merged_address'].str.lower()


In [16]:
# TO DO: just run this code snippet. Adds gsheet_df to the Google Sheet. Please check to make sure changes are okay.

gsheet_df = gsheet_df.fillna("")
worksheet = sh.worksheet(WORKSHEET_NAME)
worksheet.update([gsheet_df.columns.values.tolist()] + gsheet_df.values.tolist())

/tmp/ipykernel_72032/3690414652.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gsheet_df = gsheet_df.fillna("")


{'spreadsheetId': '1kjFa5tOKPqFlPQMpJhIoAEQmVGrjxHbCt6ng8hrZPKA',
 'updatedRange': 'Landfills!A1:Z380',
 'updatedRows': 380,
 'updatedColumns': 26,
 'updatedCells': 9880}

# Visualize Results

In [17]:
import folium

map_center = [37.5, -119.5] # Central California
m = folium.Map(location=map_center, zoom_start=6)

for _, row in gsheet_df[gsheet_df['status'] == 'true'].iterrows():
    folium.Marker(
        location=[row['closest_latitude'], row['closest_longitude']],
        popup=f"{row['closest_address_line_1']}<br>Status: {row['status']}",
        icon=folium.Icon(color='green')
    ).add_to(m)

for _, row in gsheet_df[gsheet_df['status'] == 'false'].iterrows():
    if row['closest_latitude'] != 0:
        folium.Marker(
            location=[row['closest_latitude'], row['closest_longitude']],
            popup="Geocoding Failed",
            icon=folium.Icon(color='red', icon='info-sign')
        ).add_to(m)

m